In [283]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import RidgeClassifier
from catboost import CatBoostClassifier

In [284]:
root_path = "/home/stefan/ioai-prep/kits/imprumuturi"
seed = 42

# Data

In [285]:
def prep_df(df: pd.DataFrame):
    df = df.drop(["customer_id", "loan_status"], axis=1, errors="ignore")

    cat_cols = df.select_dtypes(include='object')
    for col in cat_cols:
        dummies = pd.get_dummies(df[col], prefix=col, drop_first=True)
        df = pd.concat([df, dummies], axis=1)
        df = df.drop([col], axis=1)

    return df

In [286]:
df = pd.read_csv(f"{root_path}/train.csv")
df_train = prep_df(df)

In [287]:
df_train.head()

,age,years_employed,annual_income,credit_score,credit_history_years,savings_assets,current_debt,defaults_on_file,delinquencies_last_2yrs,derogatory_marks,...,payment_to_income_ratio,occupation_status_Self-Employed,occupation_status_Student,product_type_Line of Credit,product_type_Personal Loan,loan_intent_Debt Consolidation,loan_intent_Education,loan_intent_Home Improvement,loan_intent_Medical,loan_intent_Personal
0,40,14.1,77335,679,4.4,67,17026,0,0,0,...,0.113,True,False,False,True,False,True,False,False,False
1,18,0.0,34946,639,1.2,292,2620,0,1,0,...,0.357,False,False,False,True,False,True,False,False,False
2,29,9.4,26778,595,5.5,191,4702,1,1,0,...,0.651,False,False,False,True,False,False,False,False,True
3,27,1.5,25624,590,5.7,414,2609,0,0,0,...,0.062,False,True,False,True,True,False,False,False,False
4,18,0.0,25676,568,1.8,19,3453,1,1,1,...,0.321,False,False,False,True,False,False,False,True,False


# Models

In [289]:
X_train, X_test, y_train, y_test = train_test_split(df_train, df["loan_status"], test_size=0.2, random_state=seed)

In [290]:
def evaluate(model):
    cv = cross_val_score(model, X_train, y_train, scoring='roc_auc', cv=3, n_jobs=-1)
    cv = cv.mean() - cv.std()
    return cv.item()

In [291]:
ridge = RidgeClassifier(alpha=1)

evaluate(ridge)

0.941421928633207

In [ ]:
cb = CatBoostClassifier(random_state=seed)

evaluate(cb)

0:	total: 64.9ms	remaining: 2m 9s
0:	total: 65.9ms	remaining: 2m 11s
1:	total: 80.5ms	remaining: 1m 20s
1:	total: 84.1ms	remaining: 1m 24s
0:	total: 71.2ms	remaining: 2m 22s
2:	total: 102ms	remaining: 1m 7s
2:	total: 103ms	remaining: 1m 8s
3:	total: 115ms	remaining: 57.5s
3:	total: 119ms	remaining: 59.4s
1:	total: 95.6ms	remaining: 1m 35s
4:	total: 129ms	remaining: 51.5s
4:	total: 134ms	remaining: 53.6s
2:	total: 111ms	remaining: 1m 13s
5:	total: 150ms	remaining: 49.9s
5:	total: 150ms	remaining: 49.9s
3:	total: 126ms	remaining: 1m 3s
6:	total: 157ms	remaining: 44.7s
7:	total: 163ms	remaining: 40.5s
8:	total: 168ms	remaining: 37.3s
6:	total: 170ms	remaining: 48.3s
9:	total: 174ms	remaining: 34.6s
4:	total: 150ms	remaining: 60s
7:	total: 190ms	remaining: 47.4s
5:	total: 166ms	remaining: 55.2s
10:	total: 195ms	remaining: 35.2s
11:	total: 205ms	remaining: 34s
6:	total: 184ms	remaining: 52.3s
8:	total: 211ms	remaining: 46.7s
12:	total: 211ms	remaining: 32.2s
13:	total: 221ms	remaining: 31.3

0.9833977085460472

In [294]:
model = cb

model.fit(X_train, y_train)

0:	total: 22.1ms	remaining: 44.1s
1:	total: 49.8ms	remaining: 49.8s
2:	total: 68.4ms	remaining: 45.5s
3:	total: 82ms	remaining: 40.9s
4:	total: 93.5ms	remaining: 37.3s
5:	total: 107ms	remaining: 35.6s
6:	total: 114ms	remaining: 32.5s
7:	total: 119ms	remaining: 29.8s
8:	total: 124ms	remaining: 27.5s
9:	total: 129ms	remaining: 25.7s
10:	total: 133ms	remaining: 24.1s
11:	total: 137ms	remaining: 22.8s
12:	total: 142ms	remaining: 21.6s
13:	total: 146ms	remaining: 20.7s
14:	total: 151ms	remaining: 20s
15:	total: 156ms	remaining: 19.4s
16:	total: 161ms	remaining: 18.8s
17:	total: 166ms	remaining: 18.3s
18:	total: 171ms	remaining: 17.8s
19:	total: 176ms	remaining: 17.4s
20:	total: 183ms	remaining: 17.3s
21:	total: 191ms	remaining: 17.2s
22:	total: 197ms	remaining: 16.9s
23:	total: 202ms	remaining: 16.6s
24:	total: 206ms	remaining: 16.3s
25:	total: 211ms	remaining: 16s
26:	total: 216ms	remaining: 15.8s
27:	total: 222ms	remaining: 15.6s
28:	total: 231ms	remaining: 15.7s
29:	total: 236ms	remainin

# Submission

In [295]:
df_test_ = pd.read_csv(f"{root_path}/test.csv")
df_test = prep_df(df_test_)

In [296]:
def age_bucket(age):
    if age < 30:
        return "Young"
    if age < 60:
        return "Adult"
    return "Senior"


subtask1 = df_test_["age"].apply(age_bucket)

In [297]:
def dti_bucket(dti):
    if dti < 20:
        return "LowRisk"
    if dti < 40:
        return "MediumRisk"
    return "HighRisk"


subtask2 = df_test_["debt_to_income_ratio"].apply(dti_bucket)

In [298]:
subtask3 = (
    df_test_["current_debt"] +
    df_test_["derogatory_marks"] +
    df_test_["delinquencies_last_2yrs"]
).astype(int)

In [302]:
subtask4 = model.predict_proba(df_test)[:, 1]

In [303]:
subtasks = [
    (1, subtask1),
    (2, subtask2),
    (3, subtask3),
    (4, subtask4),
]


def build_subtask(sid, answer):
    return pd.DataFrame(
        {"subtaskID": sid, "datapointID": df_test_["customer_id"], "answer": answer}
    )


submission = pd.concat([build_subtask(sid, ans) for sid, ans in subtasks])
submission.head()

,subtaskID,datapointID,answer
0,1,CUST146767,Adult
1,1,CUST136829,Adult
2,1,CUST119409,Young
3,1,CUST112393,Young
4,1,CUST143342,Adult


In [304]:
submission.to_csv(f"{root_path}/submission.csv", index=False)